# Comparer les branches de detection

Ce notebook lance les memes cas de test centralises contre les branches activees : `generic`, `spacy`, `gliner`, `regex`.

Toute la configuration (interrupteurs de branches, chemins des modeles, labels GLiNER, ...) est centralisee dans la premiere cellule de code : modifiez-la a un seul endroit, les cellules suivantes la reutilisent. Executez les cellules dans l'ordre.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "compliance_nlp").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet depuis le dossier courant.")


ROOT = find_project_root(Path.cwd().resolve())

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENABLE_GENERIC_BRANCH = False
ENABLE_SPACY_BRANCH = False
ENABLE_GLINER_BRANCH = True
ENABLE_REGEX_BRANCH = False

MODEL_CACHE_DIR = r"C:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"C:\Workspaces\ModelStore"
SPACY_MODEL = rf"{MODEL_STORE_DIR}\fr_core_news_md"
SPACY_SYNONYMS_PATH = ROOT / "configs" / "spacy_synonyms.csv"
JEUXDEMOTS_STORE_PATH = Path(MODEL_STORE_DIR) / "jeuxdemots"
GLINER_MODEL = rf"{MODEL_STORE_DIR}\gliner_multi-v2.1"
GLINER_SOURCE_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LOCAL_FILES_ONLY = True
GLINER_LABEL_GROUPS = {
    "sante": [
        "donnee de sante",
        "maladie",
        "pathologie",
        "autisme",
        "handicap",
        "trouble du neurodeveloppement",
        "etat de sante",
        "condition medicale",
        "probleme de sante",
    ],
    "politique": [
        "ideologie politique",
        "position politique",
    ],
    "religion": ["conviction religieuse", "religion", "appartenance religieuse"],
    "syndical": ["appartenance syndicale", "engagement syndical"],
    "orientation_sexuelle": ["orientation sexuelle"],
    "origine": ["origine ethnique", "origine raciale"],
    "biometrie_genetique": ["donnee genetique", "donnee biometrique"],
    "conformite_conseil": [
        "clause beneficiaire imprecise",
        "conseil non professionnel",
        "promesse de performance",
    ],
}
GLINER_LABELS = [label for labels in GLINER_LABEL_GROUPS.values() for label in labels]

pd.set_option("display.max_colwidth", 180)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

enabled_branches = tuple(
    branch
    for branch, enabled in {
        "generic": ENABLE_GENERIC_BRANCH,
        "spacy": ENABLE_SPACY_BRANCH,
        "gliner": ENABLE_GLINER_BRANCH,
        "regex": ENABLE_REGEX_BRANCH,
    }.items()
    if enabled
)

params = {
    "enabled_branches": enabled_branches,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "spacy_model": SPACY_MODEL,
    "spacy_synonyms_path": str(SPACY_SYNONYMS_PATH),
    "gliner_model": GLINER_MODEL,
    "gliner_threshold": GLINER_THRESHOLD,
    "gliner_local_files_only": GLINER_LOCAL_FILES_ONLY,
    "gliner_label_groups": GLINER_LABEL_GROUPS,
    "gliner_labels": GLINER_LABELS,
}
params


{'enabled_branches': ('generic', 'spacy', 'gliner', 'regex'),
 'model_cache_dir': 'C:\\Workspaces\\ModelCache',
 'model_store_dir': 'C:\\Workspaces\\ModelStore',
 'spacy_model': 'C:\\Workspaces\\ModelStore\\fr_core_news_md',
 'spacy_synonyms_path': 'C:\\Workspaces\\TestPresentationNLP\\configs\\spacy_synonyms.csv',
 'gliner_model': 'C:\\Workspaces\\ModelStore\\gliner_multi-v2.1',
 'gliner_threshold': 0.5,
 'gliner_local_files_only': True,
 'gliner_label_groups': {'sante': ['donnee de sante',
   'maladie',
   'pathologie',
   'autisme',
   'handicap',
   'trouble du neurodeveloppement',
   'etat de sante',
   'condition medicale',
   'probleme de sante'],
  'politique': ['ideologie politique', 'position politique'],
  'religion': ['conviction religieuse', 'religion', 'appartenance religieuse'],
  'syndical': ['appartenance syndicale', 'engagement syndical'],
  'orientation_sexuelle': ['orientation sexuelle'],
  'origine': ['origine ethnique', 'origine raciale'],
  'biometrie_genetique':

In [52]:
import sys

# Toutes les variables de configuration (ROOT, chemins modeles, branches, labels GLiNER, ...)
# sont definies une seule fois dans la cellule precedente et reutilisees ici.
loaded_package = sys.modules.get("compliance_nlp")
if loaded_package is not None and not hasattr(loaded_package, "refresh_spacy_synonyms_from_forbidden_words"):
    for module_name in list(sys.modules):
        if module_name == "compliance_nlp" or module_name.startswith("compliance_nlp."):
            del sys.modules[module_name]

from compliance_nlp import (
    load_generic_detection_rules,
    load_whitelist_terms,
    refresh_spacy_synonyms_from_forbidden_words,
)
from compliance_nlp.pipeline import analyze_text

spacy_synonyms_count = refresh_spacy_synonyms_from_forbidden_words(
    forbidden_words_path=ROOT / "configs" / "Mots_interdits.csv",
    synonyms_path=SPACY_SYNONYMS_PATH,
    jeuxdemots_store_path=JEUXDEMOTS_STORE_PATH,
    enrich_with_jeuxdemots=True,
    allow_network=False,
    max_synonyms=5,
    min_weight=25,
)


generic_rules = load_generic_detection_rules(ROOT / "configs" / "Mots_interdits.csv")
whitelist_terms = load_whitelist_terms(ROOT / "configs" / "article9_whitelist.csv")

referentials_df = pd.DataFrame([
    {"referentiel": "generic_rules", "count": len(generic_rules)},
    {"referentiel": "whitelist_terms", "count": len(whitelist_terms)},
    {"referentiel": "spacy_synonyms", "count": spacy_synonyms_count},
    {"referentiel": "gliner_label_groups", "count": len(GLINER_LABEL_GROUPS)},
    {"referentiel": "gliner_labels", "count": len(GLINER_LABELS)},
])
referentials_df


,referentiel,count
0,generic_rules,79
1,whitelist_terms,3
2,spacy_synonyms,77
3,gliner_label_groups,8
4,gliner_labels,24


In [53]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases.csv"
TEST_CASE_COLUMNS = ["case_id", "text", "control_family", "comment"]

# sep=None + engine="python" detecte automatiquement le separateur (Excel bascule parfois , <-> ;)
test_cases = pd.read_csv(
    TEST_CASES_PATH,
    keep_default_na=False,
    sep=None,
    engine="python",
    encoding="utf-8-sig",
)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS].copy()
test_cases = test_cases[["case_id", "text", "control_family", "comment"]]
#test_cases


In [54]:
def finding_key(finding):
    return finding.rule_id or finding.code


def run_detector(text: str, case_id: str):
    return analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=generic_rules,
        whitelist_terms=whitelist_terms,
        enabled_branches=enabled_branches,
        spacy_model=SPACY_MODEL,
        spacy_synonyms_path=str(SPACY_SYNONYMS_PATH),
        gliner_model=GLINER_MODEL,
        gliner_cache_dir=MODEL_CACHE_DIR,
        gliner_source_model=GLINER_SOURCE_MODEL,
        gliner_labels=GLINER_LABEL_GROUPS,
        gliner_threshold=GLINER_THRESHOLD,
        gliner_local_files_only=GLINER_LOCAL_FILES_ONLY,
    )


def empty_row(case, engine):
    return {
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_status": "none",
        "decision": "aucune detection",
        "ignored_by_whitelist": False,
        "whitelist_expression": None,
        "whitelist_reason": None,
        "detection_engine": engine,
        "predicted_key": None,
        "code": None,
        "rule_id": None,
        "rule_scope": None,
        "regulatory_family": None,
        "section": None,
        "category": None,
        "matched_term": None,
        "detection_type": None,
        "score": None,
        "branch_score": None,
        "generic_score": None,
        "spacy_score": None,
        "gliner_score": None,
        "regex_score": None,
        "severity": None,
        "evidence": None,
    }


def finding_dict_to_row(case, finding, detection_status="active", whitelist_expression=None, whitelist_reason=None):
    return {
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_status": detection_status,
        "decision": "detecte puis ignore par whitelist" if detection_status == "ignored_by_whitelist" else "alerte active",
        "ignored_by_whitelist": detection_status == "ignored_by_whitelist",
        "whitelist_expression": whitelist_expression,
        "whitelist_reason": whitelist_reason,
        "detection_engine": finding.get("detection_engine"),
        "predicted_key": finding.get("rule_id") or finding.get("code"),
        "code": finding.get("code"),
        "rule_id": finding.get("rule_id"),
        "rule_scope": finding.get("rule_scope"),
        "regulatory_family": finding.get("regulatory_family"),
        "section": finding.get("section"),
        "category": finding.get("category"),
        "matched_term": finding.get("matched_term"),
        "detection_type": finding.get("detection_type"),
        "score": finding.get("score"),
        "branch_score": finding.get("branch_score"),
        "generic_score": finding.get("generic_score"),
        "spacy_score": finding.get("spacy_score"),
        "gliner_score": finding.get("gliner_score"),
        "regex_score": finding.get("regex_score"),
        "severity": finding.get("severity"),
        "evidence": finding.get("evidence"),
    }


def findings_to_rows(case, analysis):
    rows = []
    for finding in analysis.findings:
        rows.append(finding_dict_to_row(case, finding.to_dict()))
    for ignored in analysis.metadata.get("whitelist_ignored_findings", []):
        rows.append(
            finding_dict_to_row(
                case,
                ignored["finding"],
                detection_status="ignored_by_whitelist",
                whitelist_expression=ignored.get("whitelist_expression"),
                whitelist_reason=ignored.get("whitelist_reason"),
            )
        )
    if not rows:
        return [empty_row(case, "none")]
    return rows


In [55]:
analyses = []
rows = []
metadata_rows = []

for _, case in test_cases.iterrows():
    analysis = run_detector(case["text"], case["case_id"])
    analyses.append(analysis)
    rows.extend(findings_to_rows(case, analysis))
    metadata_rows.append({
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "enabled_branches": " | ".join(analysis.metadata.get("enabled_branches", [])),
        "raw_finding_count": analysis.metadata.get("raw_finding_count"),
        "finding_count": analysis.metadata.get("finding_count"),
        "whitelist_ignored_count": analysis.metadata.get("whitelist_ignored_count"),
        "generic_finding_count": analysis.metadata.get("generic_finding_count"),
        "spacy_finding_count": analysis.metadata.get("spacy_finding_count"),
        "gliner_finding_count": analysis.metadata.get("gliner_finding_count"),
        "regex_finding_count": analysis.metadata.get("regex_finding_count"),
        "generic_max_score": analysis.metadata.get("generic_max_score"),
        "spacy_max_score": analysis.metadata.get("spacy_max_score"),
        "gliner_max_score": analysis.metadata.get("gliner_max_score"),
        "regex_max_score": analysis.metadata.get("regex_max_score"),
        "raw_finding_count_by_engine": analysis.metadata.get("raw_finding_count_by_engine"),
        "whitelist_ignored_count_by_engine": analysis.metadata.get("whitelist_ignored_count_by_engine"),
        "branch_errors": analysis.metadata.get("branch_errors"),
    })

results_df = pd.DataFrame(rows)
case_summary_df = pd.DataFrame(metadata_rows)
#case_summary_df


In [56]:
alerts_df = results_df[results_df["predicted_key"].notna()].copy()
alerts_df.sort_values(["case_id", "detection_status", "detection_engine", "score"], ascending=[True, True, True, False])


,case_id,control_family,comment,detection_status,decision,ignored_by_whitelist,whitelist_expression,whitelist_reason,detection_engine,predicted_key,...,matched_term,detection_type,score,branch_score,generic_score,spacy_score,gliner_score,regex_score,severity,evidence
3,case_004,,,active,alerte active,False,NaN,NaN,regex,PHONE_NUMBER_FR,...,0668801470,regex,0.90,0.90,NaN,NaN,NaN,0.9,medium,"""RDV POUR ARBITRAGE VOIR SI RENDEMENT DU FOND EUROS A ETE COMMUNIQUE APPRT 1201 0668801470"""
6,case_007,,,active,alerte active,False,NaN,NaN,gliner,gliner_conseil_non_professionnel,...,Carac,entity,0.73,0.73,NaN,NaN,0.73,NaN,medium,"""RENDEZ VOUS POUR VERSEMENT RMC ET RACHAT TOTAL CEC MONSIEUR ET VERSEMENTS COMPLEMENTAIRES Faire présentation DOM + Optimiser ses impôts. Pour cet objectif, votre durée de plac..."
11,case_012,,,active,alerte active,False,NaN,NaN,gliner,gliner_religion,...,PERIN,entity,0.53,0.53,NaN,NaN,0.53,NaN,medium,"""attention PLA a surveiller 50 euros au lieu de 30 euros + migrer le PERP vers PERIN"""
15,case_016,,,active,alerte active,False,NaN,NaN,regex,PHONE_NUMBER_FR,...,0652924440,regex,0.90,0.90,NaN,NaN,NaN,0.9,medium,"""PLACEMENT SUITE SUCCESSION OUVERTURE CEPAT/REVOIR POUR REFLEXION PROPOSITION POC 0652924440 Lors de notre entretien du jour avec Madame, nous réalisons un bilan général sur sa..."
30,case_031,,,active,alerte active,False,NaN,NaN,regex,PHONE_NUMBER_FR,...,0672202686,regex,0.90,0.90,NaN,NaN,NaN,0.9,medium,"""0672202686"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1115,case_973,,,active,alerte active,False,NaN,NaN,regex,PHONE_NUMBER_FR,...,0320940737,regex,0.90,0.90,NaN,NaN,NaN,0.9,medium,"""0320940737 RDV positionné suite au décès de son époux MASSCHELEIN Pierre - 5539352 Capitaux de la RMC0331227 lui revenant : 2830.09 Mme 83 ans non adhérente CARAC ACTE DC + Do..."
1117,case_975,,,active,alerte active,False,NaN,NaN,generic,mots_interdits_052_profiteur,...,profiter,fuzzy,0.70,0.70,0.7,NaN,NaN,NaN,high,"""Je rencontre ce jour Monsieur BAZIN Xavier suite au décès de sa maman adhérente à notre mutuelle. Monsieur BAZIN étant bénéficiaire d'une partie du compte épargne carac de not..."
1118,case_975,,,active,alerte active,False,NaN,NaN,spacy,mots_interdits_052_profiteur,...,profiter,fuzzy,0.69,0.69,NaN,0.69,NaN,NaN,high,"""Je rencontre ce jour Monsieur BAZIN Xavier suite au décès de sa maman adhérente à notre mutuelle. Monsieur BAZIN étant bénéficiaire d'une partie du compte épargne carac de not..."
1121,case_978,,,active,alerte active,False,NaN,NaN,generic,mots_interdits_052_profiteur,...,profiter,fuzzy,0.70,0.70,0.7,NaN,NaN,NaN,high,"""MISE EN PLACE PLA SUITE AU RENDEZ VOUS DU JOUR A DOMICILE A LA DEMADE DU COUPLE. NOUS FAISONS LE BILAN DE LEUR SITUATION. MR SOUHAIRE REPRENDRE LES PRELEVEMENTS AUTOMATIQUES S..."


In [57]:
whitelist_ignored_df = alerts_df[alerts_df["ignored_by_whitelist"]].copy()
whitelist_ignored_df[[
    "case_id",
    "decision",
    "detection_engine",
    "predicted_key",
    "matched_term",
    "score",
    "whitelist_expression",
    "whitelist_reason",
    "evidence",
]].sort_values(["case_id", "detection_engine", "score"], ascending=[True, True, False])


,case_id,decision,detection_engine,predicted_key,matched_term,score,whitelist_expression,whitelist_reason,evidence
93,case_089,detecte puis ignore par whitelist,generic,mots_interdits_001_handicap,Handicap,0.9,epargne handicap,produit CARAC,"""Ouverture contrat Epargne handicap pour sa fille Lucie"""
94,case_089,detecte puis ignore par whitelist,spacy,mots_interdits_001_handicap,Handicap,0.9,epargne handicap,produit CARAC,"""Ouverture contrat Epargne handicap pour sa fille Lucie"""
739,case_656,detecte puis ignore par whitelist,generic,mots_interdits_001_handicap,Handicap,0.9,epargne handicap,produit CARAC,"""POUR EPARGNE HANDICAP"""
740,case_656,detecte puis ignore par whitelist,spacy,mots_interdits_001_handicap,Handicap,0.9,epargne handicap,produit CARAC,"""POUR EPARGNE HANDICAP"""


In [58]:
def join_unique_strings(values):
    cleaned = {str(value) for value in values if isinstance(value, str) and value}
    return " | ".join(sorted(cleaned))


comparison_df = (
    alerts_df.groupby(["case_id", "control_family", "comment", "detection_status", "decision", "detection_engine"], dropna=False)
    .agg(
        detection_count=("predicted_key", "size"),
        max_score=("score", "max"),
        predicted_keys=("predicted_key", join_unique_strings),
        matched_terms=("matched_term", join_unique_strings),
        whitelist_expressions=("whitelist_expression", join_unique_strings),
    )
    .reset_index()
)
#comparison_df


In [59]:
count_pivot_df = comparison_df.pivot_table(
    index=["case_id", "control_family", "comment", "detection_status", "decision"],
    columns="detection_engine",
    values="detection_count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
#count_pivot_df


In [60]:
score_columns = ["generic_score", "spacy_score", "gliner_score", "regex_score"]
score_view_df = alerts_df[[
    "case_id",
    "detection_status",
    "decision",
    "ignored_by_whitelist",
    "detection_engine",
    "predicted_key",
    "matched_term",
    "detection_type",
    *score_columns,
    "whitelist_expression",
    "whitelist_reason",
    "evidence",
]].sort_values(["case_id", "detection_status", "detection_engine", "predicted_key"], na_position="last")
#score_view_df


In [61]:
branch_errors_df = case_summary_df[["case_id", "branch_errors"]].copy()
branch_errors_df = branch_errors_df[branch_errors_df["branch_errors"].map(bool)]
branch_errors_df


,case_id,branch_errors


## Export des resultats

Cette cellule ecrit les vues principales du notebook dans `outputs/comparaison_branches` pour permettre une reprise dans Excel ou dans le notebook d'export multi-onglets.


In [62]:
export_dir = ROOT / "outputs" / "comparaison_branches"
export_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "test_cases": test_cases,
    "referentials": referentials_df,
    "case_summary": case_summary_df,
    "results_all": results_df,
    "alerts": alerts_df,
    "whitelist_ignored": whitelist_ignored_df,
    "comparison": comparison_df,
    "count_pivot": count_pivot_df,
    "score_view": score_view_df,
    "branch_errors": branch_errors_df,
}

export_paths = []
for name, dataframe in exports.items():
    path = export_dir / f"{name}.csv"
    dataframe.to_csv(path, index=False, encoding="utf-8-sig")
    export_paths.append({
        "name": name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "path": str(path),
    })

export_manifest_df = pd.DataFrame(export_paths)
export_manifest_path = export_dir / "manifest.csv"
export_manifest_df.to_csv(export_manifest_path, index=False, encoding="utf-8-sig")
export_manifest_df


,name,rows,columns,path
0,test_cases,999,4,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\test_cases.csv
1,referentials,5,2,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\referentials.csv
2,case_summary,999,18,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\case_summary.csv
3,results_all,1144,26,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\results_all.csv
4,alerts,344,26,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\alerts.csv
5,whitelist_ignored,4,26,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\whitelist_ignored.csv
6,comparison,288,11,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\comparison.csv
7,count_pivot,199,9,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\count_pivot.csv
8,score_view,344,15,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\score_view.csv
9,branch_errors,0,2,C:\Workspaces\TestPresentationNLP\outputs\comparaison_branches\branch_errors.csv
